In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv('C:\\Users\\Abdullah\\Desktop\\Code\\Sentiment Based Stock\\Sentiment Based Stocks\\results\\final_dataset_labeled.csv')
df['hour'] = pd.to_datetime(df['hour'])

price_cols     = ['open', 'high', 'low', 'close', 'volume',
                  'returns', 'volatility', 'rsi', 'ma_7', 'ma_30']
sentiment_cols = ['mean_finbert', 'mean_bullish', 'mean_bearish',
                  'mean_strength', 'post_count']


In [3]:
TARGET = 'label_3h'

train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

price_scaler     = StandardScaler()
sentiment_scaler = StandardScaler()

train_price = price_scaler.fit_transform(train[price_cols])
test_price  = price_scaler.transform(test[price_cols])
train_sent  = sentiment_scaler.fit_transform(train[sentiment_cols])
test_sent   = sentiment_scaler.transform(test[sentiment_cols])
train_regime = train['regime'].values
test_regime  = test['regime'].values
train_y = train[TARGET].values
test_y  = test[TARGET].values

In [4]:
SEQ_LEN = 24

class RegimeDataset(Dataset):
    def __init__(self, price, sent, regime, y):
        self.price, self.sent, self.regime, self.y = [], [], [], []
        for i in range(SEQ_LEN, len(price)):
            self.price.append(price[i-SEQ_LEN:i])
            self.sent.append(sent[i-SEQ_LEN:i])
            self.regime.append(regime[i])
            self.y.append(y[i])
        self.price  = torch.tensor(np.array(self.price),  dtype=torch.float32)
        self.sent   = torch.tensor(np.array(self.sent),   dtype=torch.float32)
        self.regime = torch.tensor(np.array(self.regime), dtype=torch.float32)
        self.y      = torch.tensor(np.array(self.y),      dtype=torch.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.price[idx], self.sent[idx], self.regime[idx], self.y[idx]

train_ds     = RegimeDataset(train_price, train_sent, train_regime, train_y)
test_ds      = RegimeDataset(test_price,  test_sent,  test_regime,  test_y)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
def run(model, label):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    for epoch in range(30):
        model.train()
        for p, s, r, y in train_loader:
            p, s, r, y = p.to(device), s.to(device), r.to(device), y.to(device)
            optimizer.zero_grad()
            out  = model(p, s, r)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for p, s, r, y in test_loader:
            p, s, r = p.to(device), s.to(device), r.to(device)
            probs = np.atleast_1d(model(p, s, r).cpu().numpy())
            preds = (probs > 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    print(f"\n── {label} ──")
    print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
    print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
    print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
    print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
    print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")

In [6]:
class AblationNoSentiment(nn.Module):
    def __init__(self):
        super().__init__()
        self.price_lstm = nn.LSTM(len(price_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.price_fc   = nn.Linear(128, 32)
        self.regime_gate = nn.Linear(1, 1)
        self.classifier = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))
        return self.classifier(p_emb).squeeze()

class AblationNoRegime(nn.Module):
    def __init__(self):
        super().__init__()
        self.price_lstm = nn.LSTM(len(price_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.price_fc   = nn.Linear(128, 32)
        self.sent_lstm  = nn.LSTM(len(sentiment_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.sent_fc    = nn.Linear(128, 32)
        self.classifier = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))
        s_out, _ = self.sent_lstm(sent)
        s_emb    = torch.relu(self.sent_fc(s_out[:, -1, :]))
        # Fixed equal weighting — no regime gate
        fused    = 0.5 * p_emb + 0.5 * s_emb
        return self.classifier(fused).squeeze()

class AblationNoFusionWeight(nn.Module):
    def __init__(self):
        super().__init__()
        self.price_lstm = nn.LSTM(len(price_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.price_fc   = nn.Linear(128, 32)
        self.sent_lstm  = nn.LSTM(len(sentiment_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.sent_fc    = nn.Linear(128, 32)
        self.classifier = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))
        s_out, _ = self.sent_lstm(sent)
        s_emb    = torch.relu(self.sent_fc(s_out[:, -1, :]))
        # Concatenation instead of weighted sum
        fused    = torch.cat([p_emb, s_emb], dim=1)
        return self.classifier(fused).squeeze()

In [7]:
run(AblationNoSentiment().to(device),     "Ablation 1: No Sentiment Branch (3h)")
run(AblationNoRegime().to(device),        "Ablation 2: No Regime Gate (3h)")
run(AblationNoFusionWeight().to(device),  "Ablation 3: No Fusion Weighting (3h)")


── Ablation 1: No Sentiment Branch (3h) ──
Accuracy  : 0.4580
Precision : 0.4666
Recall    : 0.4237
F1        : 0.4441
AUC       : 0.4766

── Ablation 2: No Regime Gate (3h) ──
Accuracy  : 0.5011
Precision : 0.5076
Recall    : 0.7896
F1        : 0.6180
AUC       : 0.4841

── Ablation 3: No Fusion Weighting (3h) ──
Accuracy  : 0.4966
Precision : 0.5048
Recall    : 0.7822
F1        : 0.6136
AUC       : 0.4797


In [8]:
TARGET = 'label_6h'

train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

price_scaler     = StandardScaler()
sentiment_scaler = StandardScaler()

train_price = price_scaler.fit_transform(train[price_cols])
test_price  = price_scaler.transform(test[price_cols])
train_sent  = sentiment_scaler.fit_transform(train[sentiment_cols])
test_sent   = sentiment_scaler.transform(test[sentiment_cols])
train_regime = train['regime'].values
test_regime  = test['regime'].values
train_y = train[TARGET].values
test_y  = test[TARGET].values

In [9]:
SEQ_LEN = 24

class RegimeDataset(Dataset):
    def __init__(self, price, sent, regime, y):
        self.price, self.sent, self.regime, self.y = [], [], [], []
        for i in range(SEQ_LEN, len(price)):
            self.price.append(price[i-SEQ_LEN:i])
            self.sent.append(sent[i-SEQ_LEN:i])
            self.regime.append(regime[i])
            self.y.append(y[i])
        self.price  = torch.tensor(np.array(self.price),  dtype=torch.float32)
        self.sent   = torch.tensor(np.array(self.sent),   dtype=torch.float32)
        self.regime = torch.tensor(np.array(self.regime), dtype=torch.float32)
        self.y      = torch.tensor(np.array(self.y),      dtype=torch.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.price[idx], self.sent[idx], self.regime[idx], self.y[idx]

train_ds     = RegimeDataset(train_price, train_sent, train_regime, train_y)
test_ds      = RegimeDataset(test_price,  test_sent,  test_regime,  test_y)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
def run(model, label):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    for epoch in range(30):
        model.train()
        for p, s, r, y in train_loader:
            p, s, r, y = p.to(device), s.to(device), r.to(device), y.to(device)
            optimizer.zero_grad()
            out  = model(p, s, r)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for p, s, r, y in test_loader:
            p, s, r = p.to(device), s.to(device), r.to(device)
            probs = np.atleast_1d(model(p, s, r).cpu().numpy())
            preds = (probs > 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    print(f"\n── {label} ──")
    print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
    print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
    print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
    print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
    print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")

In [11]:
class AblationNoSentiment(nn.Module):
    def __init__(self):
        super().__init__()
        self.price_lstm = nn.LSTM(len(price_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.price_fc   = nn.Linear(128, 32)
        self.regime_gate = nn.Linear(1, 1)
        self.classifier = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))
        return self.classifier(p_emb).squeeze()

class AblationNoRegime(nn.Module):
    def __init__(self):
        super().__init__()
        self.price_lstm = nn.LSTM(len(price_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.price_fc   = nn.Linear(128, 32)
        self.sent_lstm  = nn.LSTM(len(sentiment_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.sent_fc    = nn.Linear(128, 32)
        self.classifier = nn.Sequential(
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))
        s_out, _ = self.sent_lstm(sent)
        s_emb    = torch.relu(self.sent_fc(s_out[:, -1, :]))
        # Fixed equal weighting — no regime gate
        fused    = 0.5 * p_emb + 0.5 * s_emb
        return self.classifier(fused).squeeze()

class AblationNoFusionWeight(nn.Module):
    def __init__(self):
        super().__init__()
        self.price_lstm = nn.LSTM(len(price_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.price_fc   = nn.Linear(128, 32)
        self.sent_lstm  = nn.LSTM(len(sentiment_cols), 64, num_layers=2,
                                  batch_first=True, bidirectional=True, dropout=0.3)
        self.sent_fc    = nn.Linear(128, 32)
        self.classifier = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))
        s_out, _ = self.sent_lstm(sent)
        s_emb    = torch.relu(self.sent_fc(s_out[:, -1, :]))
        # Concatenation instead of weighted sum
        fused    = torch.cat([p_emb, s_emb], dim=1)
        return self.classifier(fused).squeeze()

In [12]:
run(AblationNoSentiment().to(device),     "Ablation 1: No Sentiment Branch (6h)")
run(AblationNoRegime().to(device),        "Ablation 2: No Regime Gate (6h)")
run(AblationNoFusionWeight().to(device),  "Ablation 3: No Fusion Weighting (6h)")


── Ablation 1: No Sentiment Branch (6h) ──
Accuracy  : 0.4875
Precision : 0.5188
Recall    : 0.2209
F1        : 0.3099
AUC       : 0.4721

── Ablation 2: No Regime Gate (6h) ──
Accuracy  : 0.4739
Precision : 0.4916
Recall    : 0.2965
F1        : 0.3699
AUC       : 0.4799

── Ablation 3: No Fusion Weighting (6h) ──
Accuracy  : 0.4777
Precision : 0.4914
Recall    : 0.0828
F1        : 0.1418
AUC       : 0.4608
